<a href="https://colab.research.google.com/github/emanuelmoretticosta-netizen/ic-fake-news-ptbr/blob/main/notebooks/01_baseline_tfidf_logreg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/roneysco/Fake.br-Corpus.git

Cloning into 'Fake.br-Corpus'...
remote: Enumerating objects: 28763, done.
remote: Total 28763 (delta 0), reused 0 (delta 0), pack-reused 28763 (from 1)
Receiving objects: 100% (28763/28763), 37.10 MiB | 16.26 MiB/s, done.
Resolving deltas: 100% (14129/14129), done.
Updating files: 100% (21602/21602), done.


In [2]:
import os
caminho = "Fake.br-Corpus/full_texts/true"
print(os.listdir(caminho)[:5])

['315.txt', '355.txt', '784.txt', '2700.txt', '2420.txt']


In [3]:
caminho_fake = "Fake.br-Corpus/full_texts/fake"
print(os.listdir(caminho_fake)[:5])

['315.txt', '355.txt', '784.txt', '2700.txt', '2420.txt']


In [4]:
import pandas as pd

def ler_noticias(pasta, label):
    dados = []
    for nome_arquivo in os.listdir(pasta):
        caminho_completo = os.path.join(pasta, nome_arquivo)
        with open(caminho_completo, "r", encoding="utf-8") as f:
            texto = f.read()
        dados.append({"texto": texto, "label": label})
    return dados

noticias_verdadeiras = ler_noticias("Fake.br-Corpus/full_texts/true", 0)
noticias_falsas = ler_noticias("Fake.br-Corpus/full_texts/fake", 1)

df = pd.DataFrame(noticias_verdadeiras + noticias_falsas)
print(df.shape)
print(df.head())

(7200, 2)
                                               texto  label
0  Jovem deixou 14 livros escritos à mão e cripto...      0
1  Fachin envia denúncia contra Lula, Dilma e out...      0
2  Gilmar Mendes diz que caso JBS é grande vexame...      0
3  Dilma sai em defesa de Lula e rebate suposta d...      0
4  Marielle tinha potencial para ser deputada, se...      0


In [5]:
from sklearn.model_selection import train_test_split

X_treino, X_teste, y_treino, y_teste = train_test_split(
    df["texto"], df["label"], test_size=0.2, random_state=42
)
print(len(X_treino), len(X_teste))

5760 1440


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

vetorizador = TfidfVectorizer(max_features=5000)
X_treino_vetor = vetorizador.fit_transform(X_treino)
X_teste_vetor = vetorizador.transform(X_teste)

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_treino_vetor, y_treino)

previsoes = modelo.predict(X_teste_vetor)

print("Acurácia:", accuracy_score(y_teste, previsoes))
print(classification_report(y_teste, previsoes))

Acurácia: 0.9423611111111111
              precision    recall  f1-score   support

           0       0.95      0.94      0.94       718
           1       0.94      0.95      0.94       722

    accuracy                           0.94      1440
   macro avg       0.94      0.94      0.94      1440
weighted avg       0.94      0.94      0.94      1440

